## Ingesting Data with `COPY INTO` and `SQL`

### Dropping The Table First

In [ ]:
%sql

DROP TABLE IF EXISTS ev_lab.bronze.ev_station_events;

### Create the Bronze Table without Data

In [ ]:
%sql

CREATE TABLE IF NOT EXISTS ev_lab.bronze.ev_station_events (
    event_id STRING,
    station_id STRING,
    charger_id STRING,
    event_time TIMESTAMP,
    event_type STRING,
    severity STRING,
    description STRING,
    resolved BOOLEAN
)
USING DELTA;

### Load Initial Data Seperately in a UC Volume

In [ ]:
import requests
import os

# Define the current catalog
catalog_name = "EV_lab"

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/bronze/raw_files/ev_events"

# Create the subdirectory if it doesn't exist
dbutils.fs.mkdirs(volume_base)

# List of files to download
files = ["EV_station_events.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Data_Ingestion_Unity_Catalog/Data/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Load Initial Data using `COPY INTO`

In [ ]:
%sql
COPY INTO ev_lab.bronze.ev_station_events
FROM '/Volumes/ev_lab/bronze/raw_files/ev_events/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
);

In [ ]:
%sql

SELECT * FROM ev_lab.bronze.ev_station_events

### Check Idempotency - Run `COPY INTO` again

In [ ]:
%sql
COPY INTO ev_lab.bronze.ev_station_events
FROM '/Volumes/ev_lab/bronze/raw_files/ev_events/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
);

In [ ]:
%sql

SELECT * FROM ev_lab.bronze.ev_station_events

### Count Number of Rows

In [ ]:
%sql

SELECT COUNT(*) FROM ev_lab.bronze.ev_station_events;

### Simulate New Data Arrival

In [ ]:
new_data = """event_id,station_id,charger_id,event_time,event_type,severity,description,resolved
EVT201,EV_IND_001,C-IND-001,2025-06-03 08:00:00,Alert,Medium,Voltage spike detected,false
EVT202,EV_IND_001,C-IND-002,2025-06-03 09:30:00,Maintenance,Low,Routine inspection,true
"""

with open("/Volumes/ev_lab/bronze/raw_files/ev_events/ev_events_new.csv", "w") as f:
    f.write(new_data)

### Run `COPY INTO` again

In [ ]:
%sql

COPY INTO ev_lab.bronze.ev_station_events
FROM '/Volumes/ev_lab/bronze/raw_files/ev_events/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
);

In [ ]:
%sql

SELECT * FROM ev_lab.bronze.ev_station_events

### Count Final Number of Rows

In [ ]:
%sql

SELECT COUNT(*) FROM ev_lab.bronze.ev_station_events;